In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigree','Age','Outcome']
df = pd.read_csv(url, names=cols)
print(df.shape)
print(df.head())
print(df['Outcome'].value_counts())

(768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigree  Age  Outcome  
0             0.627   50        1  
1             0.351   31        0  
2             0.672   32        1  
3             0.167   21        0  
4             2.288   33        1  
Outcome
0    500
1    268
Name: count, dtype: int64


In [2]:
# Some features have 0 which is biologically impossible → treat as NaN
zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)

# Impute with median (robust to outliers)
df.fillna(df.median(), inplace=True)

# Features and target
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale (important for logistic regression!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [3]:
# Train with L2 regularization
model = LogisticRegression(C=0.5, solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

# Classification report
print(classification_report(y_test, y_pred,
      target_names=['Non-diabetic','Diabetic']))

# AUC
auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.3f}")   # Expect ~0.83

# Cross-validation (5-fold)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
print(f"CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

              precision    recall  f1-score   support

Non-diabetic       0.75      0.80      0.77       100
    Diabetic       0.57      0.50      0.53        54

    accuracy                           0.69       154
   macro avg       0.66      0.65      0.65       154
weighted avg       0.69      0.69      0.69       154

AUC-ROC: 0.812
CV AUC: 0.838 ± 0.030


In [4]:
# Feature importance via coefficients
feature_names = cols[:-1]
coefs = model.coef_[0]
odds_ratios = np.exp(coefs)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefs.round(3),
    'Odds Ratio': odds_ratios.round(3)
}).sort_values('Coefficient', key=abs, ascending=False)

print(importance_df)

            Feature  Coefficient  Odds Ratio
1           Glucose        1.160       3.189
5               BMI        0.673       1.960
0       Pregnancies        0.370       1.447
6  DiabetesPedigree        0.230       1.259
7               Age        0.149       1.161
4           Insulin       -0.057       0.945
2     BloodPressure       -0.038       0.962
3     SkinThickness        0.034       1.035
